# COMP8410 Advanced Topics in Artificial Intelligence

## Assignment 1 – Environment Setup

**Created By:** Artittaya Klairung

### Instructions

Before starting the assignment, run the environment setup notebook (`ENV_SETUP.ipynb`) to prepare your computer for the project. This step installs the required Python packages and checks whether your system is ready to run the AI models.

The setup process includes checking Python dependencies, GPU availability, Hugging Face access, model permissions, and the required project data folders. Consequently, this setup only needs to be completed once per machine. After that, notebooks 1–5 can run independently, as each notebook defines its own shared settings in the first cell.

### Project Overview

This assignment focuses on developing and evaluating AI solutions for Entertainment Industry's K-pop comeback merchandise workflow. 
Two AI systems **(System A and System B)** are compared across four different tasks to evaluate their performance and identify the most suitable approach.

| Order | Notebook | AI Task | System A | System B |
|---|---|---|---|---|
| 1 | `01_Trend_Analysis` | Identify emerging merchandise trends | CLIP ViT-L/14 | SigLIP 2 So400m |
| 2 | `02_Design_Generation` | Generate merchandise artwork | Lumina-Image-2.0 | SDXL Base 1.0 |
| 3 | `03_Global_Pricing` | Estimate pricing tiers for new products | GAT | FAGCN |
| 4 | `04_Comment_Moderation` | Moderate launch-day comments | Granite Guardian 3.1 2B | Qwen3Guard-Gen-4B |
| 5 | `05_System_Evaluation` | Compare results and select the best system | — | — |


### Execution Order

1. **ENV_SETUP – Setup and Checks:** -> Prepare working environment before start to run all models
2. **Notebook 01 – Trend Analysis:** -> Compare CLIP and SigLIP 2 for identifying emerging merchandise trends.
3. **Notebook 02 – Design Generation:** -> Compare Lumina-Image-2.0 (flow matching) and SDXL (diffusion) for generating merchandise artwork.
4. **Notebook 03 – Global Pricing:** -> Compare GAT and FAGCN for merchandise pricing analysis.
5. **Notebook 04 – Comment Moderation:** -> Compare Granite Guardian 3.1 2B and Qwen3Guard-Gen-4B for moderating fan comments in Korean, Thai and English.
6. **Notebook 05 – System Evaluation:** -> Evaluate the experimental results and determine whether System A, System B, or a hybrid approach provides the best overall solution.


# 1. GPU Checking

I will check in all notebook before I begin, ensuring that I used a correct GPU to avoid laptop crashed issues that I have faced for now.

In [103]:
!nvidia-smi

Sat Oct 31 06:38:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [104]:
import torch

if torch.cuda.is_available():
    print("GPU Available:", torch.cuda.get_device_name(0))
    print("CUDA Version:", torch.version.cuda)
    print("GPU Count:", torch.cuda.device_count())
else:
    print("No CUDA-compatible GPU detected.")

GPU Available: Tesla T4
CUDA Version: 13.0
GPU Count: 1


> This `ENV_SETUP.ipynb` will use GPU named **Tesla T4** from Google Colab Extention to run.

# 2. Set-up Shared Configuration and Environment Settings

I create this section as the common settings used across all notebooks, including library imports, experiment configurations, project directories, random seed control, and automatic device selection (GPU or CPU from Google Colab Extention). 

As a result, These settings help maintain consistency and reproducibility throughout the experiments.



In [105]:
#import libraries
from pathlib import Path  # for folder and file path management
import os                 # for file and directory operations
import sys                # for accessing Python data and settings
import json               # for reading and managing JSON data
import time               # for measuring program execution time
import random             # for generating random numbers and data
import platform           # for checking operating system information
import warnings           # for managing warning messages

# import libraries for data analysis and visualization
import numpy as np        # for mathematical calculations and array operations
import pandas as pd       # for data manipulation, analysis, and processing

# disable unnecessary warning messages to improve Notebook readability
warnings.filterwarnings("ignore")


'''
1. Experiment Configuration 
- Set values for the experiment to ensure consistency and fairness when comparing System A and System B to conduct consistent and fair experiments.

'''

# Random seed is set to "42" for controlling random data generation, ensuring that experiments can be reproduced under the same settings, such as dataset splitting, sampling, and AI image generation.
SEED = 42


 **GPU Cost per Hour (`GPU_HOURLY_USD`)** stands for estimates the cost of running an AI model on a GPU for one hour (USD).

The reason that I would like to measure processing time alone does not reflect the financial cost of running each system. Additionally, from COMP8410 Assignment 2 - Case study feedback I relized that compute unit time cannot estimate in business impact but the number of cost will be the main factor of business decision.

**References:**
1. [Google Colab Alternatives and What a GPU Costs by the Hour (gpuperhour.com)](https://gpuperhour.com/blog/google-colab-alternatives) (prices as of 21 Sep 2026)
2. [Colab GPUs Features & Pricing (McCormick)](https://mccormickml.com/2024/04/23/colab-gpus-features-and-pricing/) (table updated March 2026)
3. [PyTorch documentation: `torch.cuda.get_device_name`](https://pytorch.org/docs/stable/generated/torch.cuda.get_device_name.html): reads the name of the GPU in use, used to select the rate automatically.

In [106]:
import torch

COLAB_PRICE_PER_UNIT_USD = 9.99 / 100   # Colab Pro: US$9.99 per 100 compute units ≈ 0.10 USD/unit

# I refered from link above as references
COLAB_UNITS_PER_HOUR_BY_GPU = {
    "T4": 1.19,
    "L4": 1.71,
    "A100": 5.40,
}

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
COLAB_UNITS_PER_HOUR = next((u for k, u in COLAB_UNITS_PER_HOUR_BY_GPU.items() if k in gpu_name), 0.0)
GPU_HOURLY_USD = COLAB_UNITS_PER_HOUR * COLAB_PRICE_PER_UNIT_USD

print(f"GPU: {gpu_name} | {COLAB_UNITS_PER_HOUR} units/hour | approximately US${GPU_HOURLY_USD:.3f}/hour")
if COLAB_UNITS_PER_HOUR == 0.0:
    print("Rate unknown for this GPU: add it to COLAB_UNITS_PER_HOUR_BY_GPU (see the Colab Resources panel).")

GPU: Tesla T4 | 1.19 units/hour | approximately US$0.119/hour


In [107]:
# Define the names of the two AI systems used in the experiment.
SYSTEMS = {
    "A": "Standard System Overview",
    # System A represents a standard system designed for general use and resource efficiency.
    "B": "Global System Overview"
    # System B represents a global system designed to support specialised applications across different tasks.
}

In [108]:

''' 
2. Reproducibility Control
- Set random seed for several libraries to contain consistent and repeatable action.
'''


def set_seed(seed: int = SEED) -> None:

    random.seed(seed) # for py

    np.random.seed(seed) # for numpy

    try:
        import torch # Deep learning

        torch.manual_seed(seed) # for CPU

        torch.cuda.manual_seed_all(seed) # for GPU

    except ImportError:
        pass

In [109]:
# Device Selection GPU from Google Colab Extention (T4)

def get_device() -> str:

    try:
        import torch

        if torch.cuda.is_available():
            return "cuda"  

    except ImportError:
        pass

    return "cpu"

set_seed()

# Display main folder directory in my device
print(" Device:", get_device())

 Device: cuda


## 3. Model Access for Notebooks (No Token Required)

Notebooks 1, 2 and 4 use seven pre-trained models from Hugging Face. All models were selected to be **open for download**, meaning no account, access token, or licence approval is required. This allows anyone to rerun the project.

**Issues that I found during this project and decide to change model to no token required**

* 4 models from the original plan required users to log in and accept a licence before downloading. 
* This will complicated if anyone will download and rerun so improving accessibility and reproducibility, I replaced with open models while maintaining the purpose of each A/B comparison.
* Furthermore, The other four models (CLIP, SigLIP 2, FLUX.1-schnell and SD 1.5) were already open for download.

| Pair | Original Model (Gated) | Replacement Model (Open) | Reason for Replacement |
|---|---|---|---|
| 2 A | FLUX.1-schnell | **Lumina-Image-2.0** | Keeps a flow-matching model (W10) for System A; 2B parameters, Apache 2.0 licence. |
| 2 B | SD 3.5 Medium | **SDXL Base 1.0** | Keeps a UNet diffusion model for System B, so the pair still compares flow matching with diffusion. |
| 4 A | Llama Guard 3 1B | **Granite Guardian 3.1 2B** | Provides a standard safety guard model for comparison. |
| 4 B | ShieldGemma 2B | **Qwen3Guard-Gen-4B** | Provides multilingual content moderation, supporting the Global Stack approach. |

**References:**
- Zhao, H. et al. (2025). *Qwen3Guard Technical Report*. arXiv:2510.14276. https://doi.org/10.48550/arXiv.2510.14276
- [Qwen3Guard on Hugging Face](https://huggingface.co/Qwen/Qwen3Guard-Gen-0.6B/commit/08b438fce4c724f862bbc8e9acba1246ec686ae1)
- [Lumina-Image-2.0 on Hugging Face](https://huggingface.co/Alpha-VLLM/Lumina-Image-2.0)
- [Diffusers documentation: Lumina2 pipeline](https://huggingface.co/docs/diffusers/api/pipelines/lumina2)
- [Stable Diffusion XL Base 1.0 on Hugging Face](https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0)
- [Granite Guardian 3.1 2B on Hugging Face](https://huggingface.co/ibm-granite/granite-guardian-3.1-2b)

In [110]:
!pip install -U huggingface_hub

In [111]:
from huggingface_hub import model_info
import pandas as pd

MODELS = [
    ("1 Trend Analysis",      "System A", "openai/clip-vit-large-patch14"),
    ("1 Trend Analysis",      "System B", "google/siglip2-so400m-patch14-384"),
    ("2 Design Generation",   "System A", "Alpha-VLLM/Lumina-Image-2.0"),
    ("2 Design Generation",   "System B", "stabilityai/stable-diffusion-xl-base-1.0"),
    ("2 Design Generation",   "Baseline", "stable-diffusion-v1-5/stable-diffusion-v1-5"),
    ("4 Comment Moderation",  "System A", "ibm-granite/granite-guardian-3.1-2b"),
    ("4 Comment Moderation",  "System B", "Qwen/Qwen3Guard-Gen-4B"),
]

rows = []
for notebook, system, mid in MODELS:
    try:
        gated = getattr(model_info(mid, token=False, files_metadata=False), "gated", False)
        status = "Open and no token needed)" if gated is False else "Gated which need login required)"
    except Exception as e:
        status = f"Not accessible: {type(e).__name__}"
    rows.append({"Notebook": notebook, "System": system, "Hugging Face Model ID": mid, "Access": status})

access = pd.DataFrame(rows).set_index(["Notebook", "System"])
access

Hugging Face Model ID  \
Notebook             System                                                  
1 Trend Analysis     System A                openai/clip-vit-large-patch14   
                     System B            google/siglip2-so400m-patch14-384   
2 Design Generation  System A                  Alpha-VLLM/Lumina-Image-2.0   
                     System B     stabilityai/stable-diffusion-xl-base-1.0   
                     Baseline  stable-diffusion-v1-5/stable-diffusion-v1-5   
4 Comment Moderation System A          ibm-granite/granite-guardian-3.1-2b   
                     System B                       Qwen/Qwen3Guard-Gen-4B   

                                                  Access  
Notebook             System                               
1 Trend Analysis     System A  Open and no token needed)  
                     System B  Open and no token needed)  
2 Design Generation  System A  Open and no token needed)  
                     System B  Open and no token needed)  
                     Baseline  Open and no token needed)  
4 Comment Moderation System A  Open and no token needed)  
                     System B  Open and no token needed)

In [112]:
problems = access[access["Access"] != "Open and no token needed)"]
assert problems.empty, f"Fix these before running notebooks 1-4:\n{problems}"
print("OK: all", len(access), "models download without a token")

OK: all 7 models download without a token


# 4. Dataset Preparation

I have different import way of Dataset which two of dataset I will manually loaded from website official directly and it don't need API key to use.

| Notebook | Dataset | Access |
|---|---|---|
| 01 Trend Analysis | Amazon Reviews 2023 | downloaded automatically in notebook 1 |
| 03 Global Pricing | KPop Online Merchandise Prices | I mannually downloaded |
| 04 Comment Moderation | BEEP! Korean comments | downloaded automatically in notebook 4 |
| 04 Comment Moderation | KOLD (optional, not used if absent) | I mannually downloaded |


**References:**
- Hou, Y., Li, J., He, Z., Yan, A., Chen, X., & McAuley, J. (2024). *Bridging Language and Items for Retrieval and Recommendation*. arXiv:2403.03952. [Amazon Reviews'23](https://amazon-reviews-2023.github.io)
- Wan, E. (2022). *KPop Online Merchandise Prices* [Dataset]. Kaggle. Licence: CC0 Public Domain. [kaggle.com/datasets/ericwan1/kpop-merchandise-prices](https://www.kaggle.com/datasets/ericwan1/kpop-merchandise-prices)
- Moon, J., Cho, W. I., & Lee, J. (2020). *BEEP! Korean Corpus of Online News Comments for Toxic Speech Detection*. SocialNLP Workshop @ ACL 2020. [github.com/kocohub/korean-hate-speech](https://github.com/kocohub/korean-hate-speech)
- Jeong, Y., Oh, J., Lee, J., Ahn, J., Moon, J., Park, S., & Oh, A. (2022). *KOLD: Korean Offensive Language Dataset*. In Proceedings of EMNLP 2022 (pp. 10818–10833). Association for Computational Linguistics. https://doi.org/10.18653/v1/2022.emnlp-main.744
- [kagglehub documentation](https://github.com/Kaggle/kagglehub)